In [1]:
from pathlib import Path
import pandas as pd

# PFR team code -> subreddit name
TEAM_SUBREDDITS = {
    "ARI": "AZCardinals",       "ATL": "falcons",
    "BAL": "ravens",            "BUF": "buffalobills",
    "CAR": "panthers",          "CHI": "CHIBears",
    "CIN": "bengals",           "CLE": "Browns",
    "DAL": "cowboys",           "DEN": "DenverBroncos",
    "DET": "detroitlions",      "GNB": "GreenBayPackers",
    "HOU": "Texans",            "IND": "Colts",
    "JAX": "Jaguars",           "KAN": "KansasCityChiefs",
    "LAC": "Chargers",          "LAR": "LosAngelesRams",
    "LVR": "raiders",           "MIA": "miamidolphins",
    "MIN": "minnesotavikings",  "NOR": "Saints",
    "NWE": "Patriots",          "NYG": "NYGiants",
    "NYJ": "nyjets",            "PHI": "eagles",
    "PIT": "steelers",          "SEA": "Seahawks",
    "SFO": "49ers",             "TAM": "buccaneers",
    "TEN": "Tennesseetitans",   "WAS": "Commanders",
}

# Draft dates shift year to year, so over-collect and let
# player-name matching do the filtering later.
DRAFT_YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
WINDOW_START, WINDOW_END = "04-20", "05-06"

BASE = "https://arctic-shift.photon-reddit.com"
RAW = Path("..") / "data" / "raw" / "reddit_threads"
RAW.mkdir(parents=True, exist_ok=True)

outcomes = pd.read_csv("../data/processed/draft_outcomes_2021_2025.csv")

assert len(TEAM_SUBREDDITS) == 32, f"expected 32, got {len(TEAM_SUBREDDITS)}"
missing = set(outcomes["team"].unique()) - set(TEAM_SUBREDDITS)
assert not missing, f"codes with no subreddit: {missing}"
print("mapping verified against draft data")

mapping verified against draft data


In [2]:
import json
import time
import requests

def fetch_threads(subreddit, year, session):
    """Return post metadata for one subreddit across one draft window."""
    resp = session.get(
        f"{BASE}/api/posts/search",
        params={
            "subreddit": subreddit,
            "after": f"{year}-{WINDOW_START}",
            "before": f"{year}-{WINDOW_END}",
            "limit": "auto",
            "sort": "desc",
            "fields": "id,title,created_utc,num_comments,score,author",
        },
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["data"]

In [3]:
session = requests.Session()
rows = []

for code, sub in TEAM_SUBREDDITS.items():
    for year in DRAFT_YEARS:
        out_path = RAW / f"{code}_{year}.json"

        if out_path.exists():
            posts = json.loads(out_path.read_text())
            status = "cached"
        else:
            try:
                posts = fetch_threads(sub, year, session)
                out_path.write_text(json.dumps(posts))
                status = "fetched"
                time.sleep(1.0)
            except Exception as exc:
                print(f"FAILED  {code} {year}: {exc}")
                continue

        rows.append({
            "team": code,
            "subreddit": sub,
            "year": year,
            "n_posts": len(posts),
            "total_comments": sum(p.get("num_comments", 0) for p in posts),
        })
        print(f"{status:8} {code} {year}  posts={len(posts):4}")

summary = pd.DataFrame(rows)
print("\ndone:", len(summary), "of", 32 * len(DRAFT_YEARS), "team-years")

fetched  ARI 2021  posts= 345
fetched  ARI 2022  posts= 262
fetched  ARI 2023  posts= 326
fetched  ARI 2024  posts= 331
fetched  ARI 2025  posts= 183
fetched  ARI 2026  posts= 268
fetched  ATL 2021  posts= 352
fetched  ATL 2022  posts= 352
fetched  ATL 2023  posts= 354
fetched  ATL 2024  posts= 368
fetched  ATL 2025  posts= 380
fetched  ATL 2026  posts= 300
fetched  BAL 2021  posts= 382
fetched  BAL 2022  posts= 378
fetched  BAL 2023  posts= 375
fetched  BAL 2024  posts= 342
fetched  BAL 2025  posts= 310
fetched  BAL 2026  posts= 351
fetched  BUF 2021  posts= 352
fetched  BUF 2022  posts= 320
fetched  BUF 2023  posts= 314
fetched  BUF 2024  posts= 304
fetched  BUF 2025  posts= 293
fetched  BUF 2026  posts= 301
fetched  CAR 2021  posts= 334
fetched  CAR 2022  posts= 336
fetched  CAR 2023  posts= 360
fetched  CAR 2024  posts= 304
fetched  CAR 2025  posts= 380
fetched  CAR 2026  posts= 300
fetched  CHI 2021  posts= 372
fetched  CHI 2022  posts= 377
fetched  CHI 2023  posts= 374
fetched  C

In [4]:
summary.pivot(index="team", columns="year", values="n_posts")

year,2021,2022,2023,2024,2025,2026
team,,,,,,
ARI,345,262,326,331,183,268
ATL,352,352,354,368,380,300
BAL,382,378,375,342,310,351
BUF,352,320,314,304,293,301
CAR,334,336,360,304,380,300
CHI,372,377,374,380,374,355
CIN,352,357,340,301,234,327
CLE,327,316,223,170,272,276
DAL,273,281,280,280,269,271


In [5]:
CANDIDATES = [
    "Redskins", "WashingtonNFL", "washingtonfootball",
    "WashingtonFootballTeam", "WashingtonFT",
]

for sub in CANDIDATES:
    try:
        posts = fetch_threads(sub, 2021, session)
        busiest = max((p.get("num_comments", 0) for p in posts), default=0)
        print(f"{sub:24} posts={len(posts):4}  busiest={busiest}")
    except Exception as exc:
        print(f"{sub:24} FAILED: {exc}")
    time.sleep(1.0)

Redskins                 posts=   0  busiest=0
WashingtonNFL            posts= 491  busiest=1139
washingtonfootball       posts=   0  busiest=0
WashingtonFootballTeam   posts=   0  busiest=0
WashingtonFT             posts=   0  busiest=0


### Handling the Washington subreddit move

Washington was the "Football Team" from 2020–2021 before becoming the
Commanders in 2022. The fanbase moved to a new subreddit rather than renaming
in place, and Reddit does not carry post history across such a move — so
`r/Commanders` returns nothing for the 2021 draft window.

The 2021 threads live in `r/WashingtonNFL` (491 posts, busiest thread 1,139
comments). Rather than complicate the main mapping, a small override table
handles team-years that deviate from it.

In [7]:
# Team-years whose subreddit differs from the current one
SUBREDDIT_OVERRIDES = {
    ("WAS", 2021): "WashingtonNFL",
}

def subreddit_for(code, year):
    """Subreddit for a team-year, honouring any override."""
    return SUBREDDIT_OVERRIDES.get((code, year), TEAM_SUBREDDITS[code])

# The empty WAS_2021.json is cached, so the harvest loop would skip it.
# Delete it to force a re-fetch.
stale = RAW / "WAS_2021.json"
if stale.exists():
    stale.unlink()

posts = fetch_threads(subreddit_for("WAS", 2021), 2021, session)
stale.write_text(json.dumps(posts))
print("WAS 2021 posts:", len(posts))

WAS 2021 posts: 220


### Consolidating harvested threads

Each team-year was saved as its own JSON file to make the collection
resumable. For analysis we want a single flat table: one row per thread,
carrying the team and year it came from.

The team code and year are recovered from the filename (`NOR_2022.json`),
which is why the naming convention was chosen deliberately.

In [8]:
records = []

for path in sorted(RAW.glob("*.json")):
    code, year = path.stem.split("_")
    for p in json.loads(path.read_text()):
        records.append({
            "team": code,
            "year": int(year),
            "post_id": p["id"],
            "title": p.get("title", ""),
            "num_comments": p.get("num_comments", 0),
            "created_utc": p.get("created_utc"),
        })

threads = pd.DataFrame(records)
print("threads:", len(threads))
print("team-years:", threads.groupby(["team", "year"]).ngroups)
threads.head()

threads: 63464
team-years: 192


,team,year,post_id,title,num_comments,created_utc
0,ARI,2021,n5tu5f,Can anyone hook up our new draft pick?,4,1620257696
1,ARI,2021,n5sxcc,Can we draft the Chargers social media manager?,14,1620255071
2,ARI,2021,n5r4vi,Here's a look at where the comp picks for 2022...,5,1620250199
3,ARI,2021,n5qpff,Are the Cardinals Super Bowl contenders?,28,1620249065
4,ARI,2021,n5o8f0,E115 - The BritishBirdgang Post-2021 NFLDraft ...,0,1620242667
